<a href="https://colab.research.google.com/github/DilaraKardas/parking_occupancy_monitoring/blob/main/parking_lot_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q ultralytics sahi

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU: ", torch.cuda.get_device_name(0))

CUDA available: True
GPU:  NVIDIA A100-SXM4-40GB


In [8]:
%pip install -q pyyaml

In [9]:
from pathlib import Path
import yaml

CONFIG_PATH = Path(
    "/content/drive/MyDrive/colab_veriler/parking_lot/config.yaml"
)
config = {
    "model": {
        "path": "yolo26s.pt",
        "device": "cuda:0"
    },
    "inference": {
        "conf": 0.5,
        "iou": 0.5,
        "imgsz": 640
    },
    "sahi": {
        "slice_height": 640,
        "slice_width": 640,
        "overlap_height_ratio": 0.2,
        "overlap_width_ratio": 0.2,
        "postprocess_type": "GREEDYNMM",
        "postprocess_match_metric": "IOS",
        "postprocess_match_threshold": 0.5
    },
    "benchmark": {
        "warmup_frames": 10,
        "measured_frames": 100
    }
}
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False, allow_unicode=True)

print(f"Configuration saved to: {CONFIG_PATH}")

Configuration saved to: /content/drive/MyDrive/colab_veriler/parking_lot/config.yaml


In [13]:
import yaml
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

MODEL_PATH = config["model"]["path"]
DEVICE = config["model"]["device"]

CONFIDENCE_THRESHOLD = config["inference"]["conf"]
IMGSZ = config["inference"]["imgsz"]
MODEL_IOU = config["inference"]["iou"]

SLICE_HEIGHT = config["sahi"]["slice_height"]
SLICE_WIDTH = config["sahi"]["slice_width"]

print("Model:", MODEL_PATH)
print("Device:", DEVICE)
print("Confidence:", CONFIDENCE_THRESHOLD)
print("Image size:", IMGSZ)
print("Model IoU:", MODEL_IOU)

Model: yolo26s.pt
Device: cuda:0
Confidence: 0.5
Image size: 640
Model IoU: 0.5


In [14]:
import csv
import time
from pathlib import Path
import yaml

import cv2
import torch
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/colab_veriler/parking_lot"
)

CONFIG_PATH = OUTPUT_DIR / "config.yaml"

VIDEO_PATH = OUTPUT_DIR / "parking_lot.mp4"

# Load settings from YAML.
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

MODEL_PATH = config["model"]["path"]

CONFIDENCE_THRESHOLD = config["inference"]["conf"]
MODEL_IOU = config["inference"]["iou"]
IMGSZ = config["inference"]["imgsz"]

SLICE_HEIGHT = config["sahi"]["slice_height"]
SLICE_WIDTH = config["sahi"]["slice_width"]
OVERLAP_HEIGHT_RATIO = config["sahi"]["overlap_height_ratio"]
OVERLAP_WIDTH_RATIO = config["sahi"]["overlap_width_ratio"]

POSTPROCESS_TYPE = config["sahi"]["postprocess_type"]
POSTPROCESS_MATCH_METRIC = config["sahi"]["postprocess_match_metric"]
POSTPROCESS_MATCH_THRESHOLD = config["sahi"]["postprocess_match_threshold"]

WARMUP_FRAMES = config["benchmark"]["warmup_frames"]
BENCHMARK_FRAMES = config["benchmark"]["measured_frames"]

# COCO class IDs: car=2, motorcycle=3, bus=5, truck=7
VEHICLE_CLASSES = {2, 3, 5, 7}

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Unique filenames for each experiment.
EXPERIMENT_ID = (
    f"{Path(MODEL_PATH).stem}"
    f"_conf{CONFIDENCE_THRESHOLD}"
    f"_iou{MODEL_IOU}"
    f"_imgsz{IMGSZ}"
)

OUTPUT_PATH = OUTPUT_DIR / f"parking_lot_boxes_{EXPERIMENT_ID}.mp4"
SUMMARY_PATH = OUTPUT_DIR / f"benchmark_summary_{EXPERIMENT_ID}.csv"
FRAME_RESULTS_PATH = OUTPUT_DIR / f"benchmark_frames_{EXPERIMENT_ID}.csv"

print("Experiment:", EXPERIMENT_ID)
print("Model:", MODEL_PATH)
print("Confidence:", CONFIDENCE_THRESHOLD)
print("Model IoU:", MODEL_IOU)
print("Image size:", IMGSZ)
print("Device:", DEVICE)


# --------------------------------------------------
# 2. HELPER FUNCTIONS
# --------------------------------------------------

def sync_device():
    """Wait for pending GPU operations before timing."""
    if DEVICE.startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def draw_fps(frame, fps_value):
    text = f"FPS: {fps_value:.1f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness, margin = 0.8, 2, 10

    (text_w, text_h), baseline = cv2.getTextSize(
        text, font, scale, thickness
    )

    x = frame.shape[1] - text_w - margin
    y = margin + text_h

    cv2.rectangle(
        frame,
        (x - 5, y - text_h - 5),
        (x + text_w + 5, y + baseline + 5),
        (0, 0, 0),
        -1,
    )

    cv2.putText(
        frame,
        text,
        (x, y),
        font,
        scale,
        (0, 255, 0),
        thickness,
        cv2.LINE_AA,
    )


def open_video_info(video_path):
    if not video_path.exists():
        raise FileNotFoundError(
            f"Video not found: {video_path}"
        )

    cap = cv2.VideoCapture(str(video_path))

    try:
        if not cap.isOpened():
            raise RuntimeError(
                f"Could not open video: {video_path}"
            )

        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    finally:
        cap.release()

    if width <= 0 or height <= 0:
        raise RuntimeError("Could not determine video dimensions.")

    if fps <= 0:
        print("Warning: Invalid source FPS. Using 30 FPS.")
        fps = 30.0

    return fps, width, height, total_frames


def save_benchmark_results(latencies, frame_numbers):
    """Save per-frame measurements and an overall summary."""

    if not latencies:
        print("No benchmark measurements available.")
        return

    total_time = sum(latencies)
    measured_count = len(latencies)

    average_latency_ms = total_time / measured_count * 1000
    average_fps = measured_count / total_time

    # Save each measured frame's latency and FPS.
    with open(
        FRAME_RESULTS_PATH, "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "frame_number",
                "latency_ms",
                "fps_from_latency",
            ],
        )
        writer.writeheader()

        for frame_number, latency in zip(frame_numbers, latencies):
            writer.writerow({
                "frame_number": frame_number,
                "latency_ms": round(latency * 1000, 3),
                "fps_from_latency": round(1 / latency, 3),
            })

    # Save overall results.
    with open(
        SUMMARY_PATH, "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=[
                "model",
                "device",
                "video",
                "warmup_frames",
                "measured_frames",
                "average_latency_ms",
                "average_fps",
                "measurement_scope",
            ],
        )
        writer.writeheader()
        writer.writerow({
            "model": MODEL_PATH,
            "device": DEVICE,
            "video": VIDEO_PATH.name,
            "warmup_frames": WARMUP_FRAMES,
            "measured_frames": measured_count,
            "average_latency_ms": round(average_latency_ms, 3),
            "average_fps": round(average_fps, 3),
            "measurement_scope": (
                "BGR-to-RGB conversion + SAHI prediction; "
                "excludes drawing and video writing"
            ),
        })

    print("\n========== BENCHMARK RESULTS ==========")
    print(f"Model: {MODEL_PATH}")
    print(f"Device: {DEVICE}")
    print(f"Measured frames: {measured_count}")
    print(f"Average latency: {average_latency_ms:.2f} ms/frame")
    print(f"Average FPS: {average_fps:.2f}")
    print(f"Summary saved: {SUMMARY_PATH}")
    print(f"Per-frame results saved: {FRAME_RESULTS_PATH}")

Experiment: yolo26s_conf0.5_iou0.5_imgsz640
Model: yolo26s.pt
Confidence: 0.5
Model IoU: 0.5
Image size: 640
Device: cuda:0


In [15]:
def main():
    cap = None
    video_writer = None

    latencies = []
    measured_frame_numbers = []

    frame_count = 0
    smoothed_fps = 0.0

    try:
        print(f"Device: {DEVICE}")

        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        else:
            print("GPU unavailable. Using CPU.")

        fps, width, height, total_frames = open_video_info(
            VIDEO_PATH
        )

        print(f"Video: {width}x{height} at {fps:.2f} FPS")
        print(f"Loading model: {MODEL_PATH}")

        detection_model = AutoDetectionModel.from_pretrained(
            model_type="ultralytics",
            model_path=MODEL_PATH,
            confidence_threshold=CONFIDENCE_THRESHOLD,
            image_size=IMGSZ,
            device=DEVICE,
        )

        # Apply the Ultralytics NMS IoU threshold.
        detection_model.model.overrides["iou"] = MODEL_IOU

        cap = cv2.VideoCapture(str(VIDEO_PATH))

        if not cap.isOpened():
            raise RuntimeError(f"Could not open: {VIDEO_PATH}")

        video_writer = cv2.VideoWriter(
            str(OUTPUT_PATH),
            cv2.VideoWriter_fourcc(*"mp4v"),
            fps,
            (width, height),
        )

        if not video_writer.isOpened():
            raise RuntimeError(
                f"Could not create output: {OUTPUT_PATH}"
            )

        print("Starting video processing...")

        while True:
            ret, frame = cap.read()

            if not ret:
                break

            # The first WARMUP_FRAMES are excluded from benchmarking.
            should_measure = (
                frame_count >= WARMUP_FRAMES
                and len(latencies) < BENCHMARK_FRAMES
            )

            if should_measure:
                sync_device()

            frame_start = time.perf_counter()

            # Convert OpenCV BGR frame to RGB for SAHI.
            frame_rgb = cv2.cvtColor(
                frame, cv2.COLOR_BGR2RGB
            )

            result = get_sliced_prediction(
                image=frame_rgb,
                detection_model=detection_model,
                slice_height=SLICE_HEIGHT,
                slice_width=SLICE_WIDTH,
                overlap_height_ratio=OVERLAP_HEIGHT_RATIO,
                overlap_width_ratio=OVERLAP_WIDTH_RATIO,
                perform_standard_pred=False,
                postprocess_type=POSTPROCESS_TYPE,
                postprocess_match_metric=POSTPROCESS_MATCH_METRIC,
                postprocess_match_threshold=POSTPROCESS_MATCH_THRESHOLD,
                verbose=0,
                progress_bar=False,
            )

            # Stop the timer after prediction, before drawing boxes.
            sync_device()
            elapsed = time.perf_counter() - frame_start

            if should_measure:
                latencies.append(elapsed)
                measured_frame_numbers.append(frame_count + 1)

            instant_fps = 1.0 / max(elapsed, 1e-9)

            smoothed_fps = (
                instant_fps
                if smoothed_fps == 0
                else 0.1 * instant_fps + 0.9 * smoothed_fps
            )

            vehicle_count = 0

            for prediction in result.object_prediction_list:
                class_id = int(prediction.category.id)

                if class_id not in VEHICLE_CLASSES:
                    continue

                x1, y1, x2, y2 = map(
                    int, prediction.bbox.to_xyxy()
                )

                score = float(prediction.score.value)
                class_name = prediction.category.name

                # Draw bounding box.
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2,
                )

                # Draw class name and confidence score.
                label = f"{class_name}: {score:.2f}"
                font = cv2.FONT_HERSHEY_SIMPLEX
                scale, thickness = 0.5, 1

                (text_w, text_h), baseline = cv2.getTextSize(
                    label, font, scale, thickness
                )

                # Keep the label background inside the frame.
                label_top = max(0, y1 - text_h - baseline - 4)

                cv2.rectangle(
                    frame,
                    (x1, label_top),
                    (x1 + text_w + 4, max(y1, text_h + baseline + 4)
                     if label_top == 0 else y1),
                    (0, 255, 0),
                    -1,
                )

                text_y = max(text_h + 2, y1 - baseline - 2)

                cv2.putText(
                    frame,
                    label,
                    (x1 + 2, text_y),
                    font,
                    scale,
                    (0, 0, 0),
                    thickness,
                    cv2.LINE_AA,
                )

                vehicle_count += 1

            draw_fps(frame, smoothed_fps)
            video_writer.write(frame)

            frame_count += 1

            print(
                f"\rFrames: {frame_count}/{total_frames} | "
                f"Vehicles: {vehicle_count} | "
                f"Benchmark: {len(latencies)}/{BENCHMARK_FRAMES}",
                end="",
                flush=True,
            )

        print("\nVideo processing finished.")

    except KeyboardInterrupt:
        print("\nProcessing interrupted.")

    except Exception as error:
        print(f"\nError ({type(error).__name__}): {error}")

    finally:
        if cap is not None:
            cap.release()

        if video_writer is not None:
            video_writer.release()

        cv2.destroyAllWindows()

        # Save even partial benchmark results if processing was interrupted.
        if latencies:
            save_benchmark_results(
                latencies, measured_frame_numbers
            )
        else:
            print(
                "No benchmark frames were measured. "
                "Check that the video contains enough frames."
            )

        print(f"Output video: {OUTPUT_PATH}")
        print("Video resources released.")
main()

Device: cuda:0
GPU: NVIDIA A100-SXM4-40GB
Video: 1920x1080 at 30.00 FPS
Loading model: yolo26s.pt
Starting video processing...
Frames: 3373/3373 | Vehicles: 44 | Benchmark: 100/100
Video processing finished.

========== BENCHMARK RESULTS ==========
Model: yolo26s.pt
Device: cuda:0
Measured frames: 100
Average latency: 139.16 ms/frame
Average FPS: 7.19
Summary saved: /content/drive/MyDrive/colab_veriler/parking_lot/benchmark_summary_yolo26s_conf0.5_iou0.5_imgsz640.csv
Per-frame results saved: /content/drive/MyDrive/colab_veriler/parking_lot/benchmark_frames_yolo26s_conf0.5_iou0.5_imgsz640.csv
Output video: /content/drive/MyDrive/colab_veriler/parking_lot/parking_lot_boxes_yolo26s_conf0.5_iou0.5_imgsz640.mp4
Video resources released.
